In [ ]:
import strahler
from importlib.metadata import version as package_version

package = package_version("strahler")
library = strahler.version()
print(f"strahler package {package}; libstrahler {library}")
if package != library:
    raise RuntimeError("Package/library version mismatch. Reinstall strahler in this kernel, "
                       "check STRAHLER_LIB, and restart the kernel before continuing.")

from notebook_support import HERE, BACKENDS, metrics_table, table
from strahler.widgets import GameView, TreeView, Stepper
from IPython.display import display

# Step through lifting

Inspect a Strahler host, adaptive t, and iterative shaving (`+iter`). Each
Stepper badge calls the public partial label materializer before checking the
candidate. Structural problems from the materializer and validity failures from
`check()` are separate diagnostics. A candidate can fail before convergence;
unresolved vertices remain undecided. Root S labels represent TOP and are not
part of the final certified region.

In [ ]:
from otj22 import hk, hk_coordinates
game = strahler.Game.from_text(hk(2))
configurations = {
    "Strahler": {"tree_k": 2},
    "adaptive t": {"adaptive_t": True},
    "adaptive t +iter": {"adaptive_t": "incremental", "iter_shave": True},
}
steppers = {}
for name, options in configurations.items():
    stepper = Stepper(strahler.trace(game, "lifting", **options), hk_coordinates(2))
    steppers[name] = stepper
    display(stepper)
    for kind in ("lift", "round_end", "phase"):
        stepper.seek(0)
        stepper.next(kind)
        for player in range(2):
            verdict = stepper.verify(player)
            print(name, kind, player, verdict)
    stepper.end()
    assert stepper.run.result.complete
    assert all(stepper.verify(p).ok for p in range(2))
metrics_table({name: stepper.run.result for name, stepper in steppers.items()})

In [ ]:
# Explicit public materializer example: S fibres with empty child cores.
partial_game = strahler.Game.from_text('0 2 1 1; 1 1 1 1; 2 1 1 2;')
labels = dict(player=0, h=2, tree="0|1|", labels=[
    dict(vertex=1, path=[0], part="S"), dict(vertex=2, path=[1], part="S")])
partial = strahler.materialize_partial(partial_game, labels)
assert [problem["condition"] for problem in partial["problems"]] == [22, 22]
print("Structural problems:", partial["problems"])
print("Independent verdict:", strahler.check(partial_game, partial["tree"], 0))
view = GameView(partial_game)
display(view, TreeView(partial["tree"], view))